In [1]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
JSON_FILE = "UtilityRates.json"

KNOWN_PLAN_NAMES = {"TOU", "Tiered", "Demand", "Flat"}


def normalize_value(v):
    """
    Convert JSON values like 'NA' to None so pandas/PostgreSQL
    can treat them as nulls.
    """
    if isinstance(v, str) and v.strip().upper() == "NA":
        return None
    return v


def get_months(d):
    """
    Some records use 'Months', one uses 'Month'.
    Return a normalized list.
    """
    if not isinstance(d, dict):
        return []
    months = d.get("Months", d.get("Month", []))
    return months if isinstance(months, list) else []


def is_explicit_plan_container(commodity_payload):
    """
    Determines whether a commodity block contains explicit plan names
    like TOU/Tiered/Demand/Flat, or whether it should be treated as
    a single 'Standard' plan (common for Gas).
    """
    plan_keys = [k for k in commodity_payload.keys() if k != "URL"]
    return any(k in KNOWN_PLAN_NAMES for k in plan_keys)


with open(JSON_FILE, "r", encoding="utf-8") as f:
    utility_data = json.load(f)

# Table storage (lists of dicts -> DataFrames)
states_rows = []
commodities_rows = []
rate_plans_rows = []
rate_components_rows = []
rate_component_months_rows = []
thresholds_rows = []
threshold_months_rows = []
source_urls_rows = []

# Surrogate key lookup maps
state_id_map = {}
commodity_id_map = {}

# Counters
next_state_id = 1
next_commodity_id = 1
next_plan_id = 1
next_component_id = 1
next_threshold_id = 1


# Seed commodity table (you already know these from the JSON)
for commodity_name in ["Gas", "Elec"]:
    commodity_id_map[commodity_name] = next_commodity_id
    commodities_rows.append({
        "commodity_id": next_commodity_id,
        "commodity_name": commodity_name
    })
    next_commodity_id += 1


# Parse JSON into normalized relational-style tables
for state_name, state_payload in utility_data.items():
    # 1) states
    state_id = next_state_id
    state_id_map[state_name] = state_id
    states_rows.append({
        "state_id": state_id,
        "state_name": state_name
    })
    next_state_id += 1

    # 2) source_urls
    url_payload = state_payload.get("URL", {})
    for commodity_name, source_url in url_payload.items():
        if commodity_name in commodity_id_map:
            source_urls_rows.append({
                "source_id": len(source_urls_rows) + 1,
                "state_id": state_id,
                "commodity_id": commodity_id_map[commodity_name],
                "source_url": source_url
            })

    # 3) commodities within state
    for commodity_name in ["Gas", "Elec"]:
        if commodity_name not in state_payload:
            continue

        commodity_payload = state_payload[commodity_name]
        commodity_id = commodity_id_map[commodity_name]

        # Decide whether we have explicit plans (TOU/Tiered/Flat/...)
        # or a single implicit "Standard" plan
        if is_explicit_plan_container(commodity_payload):
            plan_items = [
                (plan_name, plan_payload)
                for plan_name, plan_payload in commodity_payload.items()
                if plan_name in KNOWN_PLAN_NAMES
            ]
        else:
            # e.g. Gas blocks like California/Wisconsin/Texas
            plan_items = [("Standard", commodity_payload)]

        for plan_name, plan_payload in plan_items:
            
            # rate_plans
            fixed_charge = normalize_value(plan_payload.get("Fixed"))
            fixed_charge_unit = plan_payload.get("Fixed_Unit")

            # consumption unit may be at plan-level OR nested inside Consumption
            consumption_unit = plan_payload.get("Consumption_Unit")
            if consumption_unit is None and isinstance(plan_payload.get("Consumption"), dict):
                consumption_unit = plan_payload["Consumption"].get("Consumption_Unit")

            tou_start = normalize_value(plan_payload.get("TOUstart"))
            tou_end = normalize_value(plan_payload.get("TOUend"))
            tou_days = normalize_value(plan_payload.get("TOUdays"))

            plan_id = next_plan_id
            rate_plans_rows.append({
                "plan_id": plan_id,
                "state_id": state_id,
                "commodity_id": commodity_id,
                "plan_name": plan_name,
                "fixed_charge": fixed_charge,
                "fixed_charge_unit": fixed_charge_unit,
                "consumption_unit": consumption_unit,
                "tou_start_hour": tou_start,
                "tou_end_hour": tou_end,
                "tou_days": tou_days,
                "notes": None
            })
            next_plan_id += 1


            # rate_components
            consumption = plan_payload.get("Consumption", {})

            if isinstance(consumption, dict):
                for comp_name, comp_val in consumption.items():
                    # Skip nested metadata keys
                    if comp_name == "Consumption_Unit":
                        continue

                    # Case 1: scalar component
                    # e.g. Rate = 0.17203, T1 = 0.42009, Peak = 0.232382
                    if not isinstance(comp_val, dict):
                        component_id = next_component_id
                        rate_components_rows.append({
                            "component_id": component_id,
                            "plan_id": plan_id,
                            "component_name": comp_name,
                            "season_name": None,
                            "rate_value": normalize_value(comp_val),
                            "rate_unit": consumption_unit
                        })
                        next_component_id += 1

                    # Case 2: nested seasonal structure
                    # e.g. Peak -> Summer/Winter -> Rate + Months
                    else:
                        for season_name, season_payload in comp_val.items():
                            if not isinstance(season_payload, dict):
                                continue

                            if "Rate" in season_payload:
                                component_id = next_component_id
                                rate_components_rows.append({
                                    "component_id": component_id,
                                    "plan_id": plan_id,
                                    "component_name": comp_name,
                                    "season_name": season_name,
                                    "rate_value": normalize_value(season_payload.get("Rate")),
                                    "rate_unit": consumption_unit
                                })
                                next_component_id += 1

                                # rate_component_months
                                for m in get_months(season_payload):
                                    rate_component_months_rows.append({
                                        "component_id": component_id,
                                        "month_num": int(m)
                                    })

            # thresholds
            # Examples:
            #   Threshold
            #   T1_Threshold
            #
            # Shape:
            # {
            #   "Summer": {"Allowance": 0.56, "Months": [...]},
            #   "Winter": {"Allowance": 7.5, "Months": [...]},
            #   "Threshold_Unit": "kWh/Day"
            # }
            threshold_keys = [k for k in plan_payload.keys() if "Threshold" in k]

            for threshold_name in threshold_keys:
                threshold_payload = plan_payload.get(threshold_name, {})
                if not isinstance(threshold_payload, dict):
                    continue

                allowance_unit = threshold_payload.get("Threshold_Unit")

                for season_name, season_payload in threshold_payload.items():
                    if season_name == "Threshold_Unit":
                        continue

                    if isinstance(season_payload, dict) and "Allowance" in season_payload:
                        threshold_id = next_threshold_id
                        thresholds_rows.append({
                            "threshold_id": threshold_id,
                            "plan_id": plan_id,
                            "threshold_name": threshold_name,
                            "season_name": season_name,
                            "allowance_value": normalize_value(season_payload.get("Allowance")),
                            "allowance_unit": allowance_unit
                        })
                        next_threshold_id += 1

                        # threshold_months
                        for m in get_months(season_payload):
                            threshold_months_rows.append({
                                "threshold_id": threshold_id,
                                "month_num": int(m)
                            })


# Create DataFrames
states_df = pd.DataFrame(states_rows)
commodities_df = pd.DataFrame(commodities_rows)
rate_plans_df = pd.DataFrame(rate_plans_rows)
rate_components_df = pd.DataFrame(rate_components_rows)
rate_component_months_df = pd.DataFrame(rate_component_months_rows)
thresholds_df = pd.DataFrame(thresholds_rows)
threshold_months_df = pd.DataFrame(threshold_months_rows)
source_urls_df = pd.DataFrame(source_urls_rows)

# Optional: sort for readability
states_df = states_df.sort_values("state_id").reset_index(drop=True)
commodities_df = commodities_df.sort_values("commodity_id").reset_index(drop=True)
rate_plans_df = rate_plans_df.sort_values("plan_id").reset_index(drop=True)
rate_components_df = rate_components_df.sort_values("component_id").reset_index(drop=True)
rate_component_months_df = rate_component_months_df.sort_values(["component_id", "month_num"]).reset_index(drop=True)
thresholds_df = thresholds_df.sort_values("threshold_id").reset_index(drop=True)
threshold_months_df = threshold_months_df.sort_values(["threshold_id", "month_num"]).reset_index(drop=True)
source_urls_df = source_urls_df.sort_values(["state_id", "commodity_id"]).reset_index(drop=True)

# Store in a dictionary
tables = {
    "states": states_df,
    "commodities": commodities_df,
    "rate_plans": rate_plans_df,
    "rate_components": rate_components_df,
    "rate_component_months": rate_component_months_df,
    "thresholds": thresholds_df,
    "threshold_months": threshold_months_df,
    "source_urls": source_urls_df,
}

# Preview
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.head(10))
    print(f"shape = {df.shape}")
